# Step 1: Building the RAG Pipeline
### Project: Evaluating the Impact of RAG on Reducing LLM Hallucinations

In this notebook, we set up the core retrieval pipeline:
1. Loading the 51 space science documents
2. Splitting them into overlapping chunks so facts don't get cut in half
3. Generating 384-dimensional vector embeddings with `all-MiniLM-L6-v2`
4. Building an in-memory cosine similarity search index and testing retrieval

In [ ]:
# Install requirements if running in Google Colab
# !pip install sentence-transformers pandas numpy matplotlib seaborn requests python-dotenv

import sys
from pathlib import Path

# Add parent directory to path so we can import from src
sys.path.append('..')

from src.data_loader import load_documents, RecursiveCharacterTextSplitter
from src.vector_store import SimpleVectorStore, EmbeddingEngine
from src.config import DOCUMENTS_DIR, VECTOR_STORE_DIR

print("Environment initialized successfully.")

## 1. Load Domain Knowledge Corpus
We load 51 factual articles covering historic and modern space missions (Apollo, Curiosity, JWST, Chandrayaan, etc.).

In [ ]:
documents = load_documents(DOCUMENTS_DIR)
print(f"Total loaded documents: {len(documents)}")
print(f"Sample document source: {documents[0].metadata['source']}")
print("Sample content preview:\n", documents[0].page_content[:280], "...")

## 2. Splitting Text into Semantic Chunks
We break the articles into 550-character chunks with a 90-character overlap. The overlap ensures that multi-word phrases and numerical parameters spanning sentence boundaries are preserved.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=550, chunk_overlap=90)
chunks = splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")
print(f"Sample chunk metadata: {chunks[0].metadata}")
print(f"\nSample chunk content:\n{chunks[0].page_content}")

## 3. Embedding Generation & Vector Indexing
We embed all chunks using `sentence-transformers/all-MiniLM-L6-v2` and index them in an in-memory normalized cosine search store.

In [ ]:
embedding_engine = EmbeddingEngine()
vector_store = SimpleVectorStore(embedding_engine=embedding_engine)
vector_store.add_documents(chunks)
vector_store.save(VECTOR_STORE_DIR)
print(f"Vector store indexed with {len(vector_store.documents)} chunks and saved to disk.")

## 4. Testing Semantic Similarity Retrieval
Let's test retrieving the top 3 most relevant chunks for a specific test query.

In [ ]:
query = "What instrument on Perseverance demonstrated producing oxygen from the Mars atmosphere?"
results = vector_store.similarity_search_with_score(query, k=3)

print(f"Query: {query}")
print("=" * 70)
for idx, (doc, score) in enumerate(results, 1):
    print(f"\n[Chunk {idx}] Similarity: {score:.4f} | Source: {doc.metadata.get('source')}")
    print(doc.page_content)